# Agent Initialization & Prompt Engineering
Noam S

In [ ]:
%pip install langchain-ollama


In [ ]:
import os
from langchain_ollama import ChatOllama
from langchain.agents.factory import create_agent
from langgraph.checkpoint.memory import MemorySaver


In [ ]:
# This function initializes the financial agent local ChatOllama model (LLaMA 3.2) with the specified tools.
def initialize_financial_agent(tools_list, prefer_gpu=True, model_name="llama3.2"):
    print(f"Initializing Local LLM core via Ollama ({model_name})...")
    llm = None
    if prefer_gpu:
        try:
            print(f"Attempting to start Ollama on GPU (num_gpu=1) with model: {model_name}...")
            llm = ChatOllama(model=model_name, temperature=0.1, num_gpu=1)
            llm.invoke("Hi")
            print("GPU initialization and model loading succeeded.")
        except Exception as e:
            print(f"GPU startup failed, falling back to CPU. Error: {str(e)}")
            llm = None
    if llm is None:
        try:
            print(f"Starting Ollama on CPU (num_gpu=0) with model: {model_name}...")
            llm = ChatOllama(model=model_name, temperature=0.1, num_gpu=0)
            llm.invoke("Hi")
            print("CPU model loading succeeded.")
        except Exception as e:
            print(f"CPU model loading failed. Error: {str(e)}")
            llm = None

    # Fallback to llama3.2:1b if 3B fails
    if llm is None and model_name != "llama3.2:1b":
        try:
            print("Attempting to load lightweight fallback model: llama3.2:1b (requires ~1.3 GB RAM)...")
            llm = ChatOllama(model="llama3.2:1b", temperature=0.1)
            llm.invoke("Hi")
            print("Lightweight fallback model loaded successfully.")
        except Exception as fe:
            print(f"Fallback to llama3.2:1b failed too. Error: {str(fe)}")
            llm = None

    if llm is None:
        raise RuntimeError("CRITICAL: Failed to initialize any Ollama local models. Please run 'ollama pull llama3.2'")

    # Detailed system prompt to guide the agent's behavior and routing decisions
    prompt = (
        "You are an elite financial AI assistant specialized in NVIDIA (NVDA) financial reports and stock analysis.\n"
        "You MUST call tools to retrieve information! Never try to explain how to use a tool instead of actually calling it!\n"
        "  - If the user asks about financial reports, revenue, net income, or historical data, you MUST call 'query_financial_reports'.\n"
        "  - If the user asks about the stock price or asks to check it, you MUST call 'get_nvidia_stock_price'.\n"
        "  - If the stock price change is 2% or more, you MUST call 'send_email_alert'.\n"
        "  - If you need to calculate a percentage change or growth between two financial numbers, you MUST call 'calculate_percentage_change' with the exact old and new values. NEVER calculate percentages manually!\n"
        "CRITICAL TRUTHFULNESS & PERIOD MATCHING INSTRUCTIONS:\n"
        "1. NEVER make up, invent, guess, or hallucinate financial figures! Use ONLY the exact figures retrieved by your tools.\n"
        "2. When the user asks about a specific quarter (e.g. Q2 2025 or Q3 2025), you MUST verify that the figures you use are from the correct period. The LEFT column is 2025 (current year), and the RIGHT column is 2024 (prior year). Make sure to read the LEFT-most column (ended in 2025) for 2025 quarters!\n"
        "3. Strictly map the quarters to their respective source files and call 'query_financial_reports' to retrieve the actual numbers:\n"
        "   - Q1 2025 calendar (ended April 27, 2025) corresponds to Q1 of Fiscal Year 2026, located in 'first_q_25.pdf'.\n"
        "   - Q2 2025 calendar (ended July 27, 2025) corresponds to Q2 of Fiscal Year 2026, located in 'second_q_25.pdf'.\n"
        "   - Q3 2025 calendar (ended October 26, 2025) corresponds to Q3 of Fiscal Year 2026, located in 'third_q_25.pdf'.\n"
        "   - FY 2025 annual (ended January 25, 2026) corresponds to Fiscal Year 2026, located in 'annual_25.pdf'.\n"
        "   You MUST query the tool to retrieve the actual revenues. Never hardcode, guess, or assume the values. Note that 2025 current period is the LEFT column, and 2024 prior period is the RIGHT column. NEVER mix up the 2025 and 2024 columns!\n"
        "4. CRITICAL FORMATTING & CONVERSION RULES:\n"
        "   - NEVER write redundant double-format parenthesis like '$46.74 billion ($46,743 million)'. This is extremely confusing and redundant! Present numbers cleanly in EITHER billions (e.g. '$46.74 billion' or '$46.7B') or millions (e.g. '$46,743 million'), but NEVER both together in parentheses!\n"
        "   Always refer to these quarters using BOTH calendar and fiscal terms, for example: 'Q2 2025 (Fiscal 2026)' or 'Q3 2025 (Fiscal 2026)'.\n"
        "5. STRICT ARITHMETIC RULE:\n"
        "   - Large Language Models are text predictors and cannot perform high-precision division or percentage calculations. You MUST call 'calculate_percentage_change' whenever you need to determine a percentage change. NEVER guess, estimate, or perform mental math for percentages!\n"
        "Write your final answer in English. Keep it extremely concise (1-2 sentences maximum, under 45 words) for real-time speed.\n"
    )

    print("Configuring tool-calling architecture with MemorySaver checkpointer...")
    checkpointer = MemorySaver()
    agent = create_agent(llm, tools_list, system_prompt=prompt, checkpointer=checkpointer)

    print("Local Financial Agent brain successfully initialized with Memory Saver.")
    return agent
